# 01 · Transfer learning for image classification (guided)

**Style A: syntax gaps.** 8 gaps (`# TODO(gN)` + `____`), each followed by a check.
Solutions: `../solutions/guided/01_transfer_classification.md`.

**Traces to** `notes/ioai-task-patterns.md` P3, P4. T25-G-SPEECH fine-tuned a torchvision `resnet18`
(swapping `conv1`/`fc`), and T26-DOUBLE loaded ResNet18 from local weights. Neither baseline used the modern
`weights=` enum or the transforms v2 API, so this notebook covers both. Syllabus: image classification,
pre-trained vision encoders, augmentation.

Data: EuroSAT (Sentinel-2 satellite tiles, 64×64 RGB, 10 land-use classes). Runtime: about 2 minutes on a T4.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models
from torchvision.transforms import v2
from datasets import load_dataset
from sklearn.metrics import confusion_matrix
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

ds = load_dataset("blanchon/EuroSAT_RGB")
NAMES = ds["train"].features["label"].names
def balanced(split, per_class, seed):
    lab = np.array(split["label"]); rng = np.random.default_rng(seed)
    return split.select(np.concatenate([rng.choice(np.flatnonzero(lab == c), per_class, replace=False) for c in range(10)]))
train, val = balanced(ds["train"], 150, 0), balanced(ds["validation"], 50, 1)
print(len(train), len(val), NAMES)

## 1. Transforms v2

`ToImage()` gives a `uint8` tensor in 0..255. Pretrained weights expect float **in [0, 1]**, normalised with the
ImageNet mean and std stored in the weights' own transform.

In [ ]:
weights = models.ResNet18_Weights.DEFAULT
MEAN, STD = weights.transforms().mean, weights.transforms().std
SIZE = 112
train_tf = v2.Compose([
    v2.ToImage(),
    v2.RandomResizedCrop(SIZE, scale=(0.6, 1.0), antialias=True),
    v2.RandomHorizontalFlip(), v2.RandomVerticalFlip(),              # satellite tiles have no "up"
    # TODO(g1): convert uint8 0..255 to float32 0..1
    v2.ToDtype(____),
    # TODO(g2): normalise with the weights' ImageNet statistics
    ____,
])
eval_tf = v2.Compose([v2.ToImage(), v2.Resize(SIZE, antialias=True), v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEAN, STD)])

In [ ]:
x = train_tf(train[0]["image"])
assert x.dtype == torch.float32 and x.shape == (3, SIZE, SIZE)
assert -3 < x.min() < 0 < x.max() < 3, "values look un-normalised (or unscaled)"
print("g1, g2 ok")

## 2. Hugging Face dataset → DataLoader

`set_transform` applies a function lazily, per batch, when rows are accessed.

In [ ]:
def with_tf(tf):
    return lambda batch: {"x": [tf(im.convert("RGB")) for im in batch["image"]], "y": batch["label"]}
train.set_transform(with_tf(train_tf)); val.set_transform(with_tf(eval_tf))

def collate(rows):
    # TODO(g3): stack the per-image tensors into (B, 3, H, W) and the labels into an int64 tensor
    return ____

train_loader = DataLoader(train, batch_size=64, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val, batch_size=128, collate_fn=collate)

In [ ]:
xb, yb = next(iter(train_loader))
assert xb.shape == (64, 3, SIZE, SIZE) and yb.dtype == torch.int64
print("g3 ok")

## 3. A pretrained ResNet18 with a new head

In [ ]:
model = models.resnet18(weights=weights)
# TODO(g4): replace the 1000-class ImageNet layer with a fresh 10-class layer (keep its input size)
model.fc = ____
model = model.to(device)

In [ ]:
assert model(xb.to(device)).shape == (64, 10)
print("g4 ok")

## 4. Linear probe first (backbone frozen), then full fine-tuning with mixed precision

In [ ]:
def tta_logits(model, xb):
    # TODO(g5): test-time augmentation: average with the logits of the horizontally flipped batch (NCHW: width is dim 3)
    return (model(xb) + model(____)) / 2

@torch.no_grad()
def evaluate(model, loader, tta=False):
    model.eval(); preds, ys = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        logits = tta_logits(model, xb) if tta else model(xb)
        preds.append(logits.argmax(1).cpu()); ys.append(yb)
    return torch.cat(preds), torch.cat(ys)

def run(model, epochs, params, lr):
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")
    for ep in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
                loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad()
            # TODO(g6): the GradScaler versions of backward / step, plus the scale update
            ____
        p, y = evaluate(model, val_loader)
        print(f"  epoch {ep}: loss {loss.item():.3f}  val acc {(p == y).float().mean():.4f}")

# TODO(g7): freeze every parameter, then unfreeze only the new head
____
print("linear probe:")
run(model, 2, model.fc.parameters(), 3e-3)

In [ ]:
assert [n for n, p in model.named_parameters() if p.requires_grad] == ["fc.weight", "fc.bias"]
print("g6, g7 ok")

In [ ]:
model.requires_grad_(True)
print("full fine-tune:")
run(model, 3, model.parameters(), 5e-4)
p, y = evaluate(model, val_loader)
p_tta, _ = evaluate(model, val_loader, tta=True)
print("val accuracy:", (p == y).float().mean().item(), "| with flip TTA:", (p_tta == y).float().mean().item())

In [ ]:
assert (p == y).float().mean() > 0.85
xt = torch.arange(6.0).view(1, 1, 2, 3)
assert torch.equal(tta_logits(lambda z: z[:, 0, 0, :], xt), torch.tensor([[1.0, 1.0, 1.0]])), "flip the WIDTH axis"
print("g5 ok")

## 5. Where are the errors? Per-class accuracy

In [ ]:
cm = confusion_matrix(y, p_tta, labels=range(10))
# TODO(g8): per-class accuracy (recall) = correct predictions of a class / number of samples of that class
per_class = ____
for name, acc in sorted(zip(NAMES, per_class), key=lambda t: t[1]):
    print(f"{name:24s} {acc:.2f}")

In [ ]:
assert np.isclose(per_class[0], ((y == 0) & (p_tta == 0)).sum().item() / (y == 0).sum().item())
print("g8 ok, all gaps done")